# xv6: Trap Handling in User Mode and Kernel Mode

## 1. What is `stvec`?

`stvec` stands for **Supervisor Trap Vector Base Address Register**.

Imagine the CPU is executing instructions when an interrupt suddenly occurs.

The CPU needs to know:

**Where should I jump to handle this interrupt?**

Answer: The address specified by `stvec`.

In xv6, `stvec` points to the entry code responsible for handling traps in Supervisor Mode.

**Important:** `stvec` specifies *where to go*, not *whether interrupts are enabled*.

---

## 2. Two Trap Paths in xv6

xv6 has two different trap paths, depending on whether the CPU was executing User Code or Kernel Code.

| | Trap from User Mode | Trap from Kernel Mode |
|---|---|---|
| `stvec` points to | `uservec` | `kernelvec` |
| Assembly entry | `trampoline.S` | `kernelvec.S` |
| Switch page table? | Yes, User → Kernel | No, already using Kernel Page Table |
| Switch stack? | Yes, User Stack → Process Kernel Stack | No, already using Kernel Stack |
| C handler | `usertrap()` | `kerneltrap()` |

### Why does `stvec` have two different values?

Because xv6 **dynamically updates `stvec`** depending on whether the CPU is running User Code or Kernel Code.

Example: A User Process makes a system call.

1. Before returning to User Mode, `usertrapret()` sets `stvec` to `uservec`.
2. The User Process executes `ecall`, triggering a trap.
3. The CPU jumps to `uservec`.
4. `uservec` switches to the Kernel Page Table and the Process's Kernel Stack, then jumps to `usertrap()`.
5. `usertrap()` sets `stvec` back to `kernelvec`.
6. If another trap occurs while the Kernel is executing, the CPU will enter `kernelvec`.

Note: The CPU does not automatically switch the Page Table or Stack when entering Supervisor Mode. xv6 performs these operations in software.

---

## 3. Why Can the Kernel Be Interrupted Too?

A trap is not necessarily a request from a User Process.

For example:

- **System call:** A process explicitly asks the Kernel to do something.
- **Hardware interrupt:** A hardware event, such as a timer interrupt, requires CPU attention.
- **Exception:** An instruction causes an exceptional condition, such as an invalid memory access.

A hardware interrupt can occur while the CPU is executing either User Code or Kernel Code, as long as the relevant interrupt is enabled.

### Key Mental Model: Kernel Work Can Still Belong to a Process

Suppose Process A is running.

1. Process A executes its own instructions in User Mode.
2. Process A calls `read()`, entering Kernel Mode.
3. The Kernel performs work **on behalf of** Process A.
4. A Timer Interrupt occurs while the Kernel **is still working for A**.
5. The Kernel may call `yield()` to let the Scheduler choose another process.

**Even when the Kernel is executing code for Process A, the CPU is still running in Process A's execution context.**

| CPU is executing | CPU Mode | Current Process |
|---|---|---|
| Process A's user code | User | A |
| Kernel handling A's `read()` | Kernel | A |
| Kernel handling an interrupt during A's execution | Kernel | A |
| `scheduler()` selecting a process | Kernel | None (`c->proc == 0`) |
| Kernel handling B's `write()` | Kernel | B |

The important distinction:

**CPU Mode and Current Process are two independent concepts.**

- CPU Mode answers: *What privileges does the CPU currently have?*
- Current Process answers: *Whose execution context is the CPU currently running?*

The CPU may switch between User Mode and Kernel Mode without switching to another process.

---

## 4. Timer Interrupts and `yield()`

In xv6, `devintr()` returns:

| Return Value | Meaning |
|---|---|
| `0` | No recognized device interrupt |
| `1` | Other device interrupt, such as UART or VirtIO |
| `2` | Timer interrupt |

Inside `kerneltrap()`:

```c
// Give up the CPU if this is a timer interrupt.
if (which_dev == 2 && myproc() != 0)
    yield();
```

Let's break it down.

**`which_dev == 2`**

Was this a Timer Interrupt?

**`myproc() != 0`**

Is the CPU currently running a Process?

**`yield()`**

Let the current Process give up the CPU so the Scheduler can select another runnable Process.

The Process is marked `RUNNABLE`, not `SLEEPING`, because it is still ready to execute.

### What Happens If the Kernel Is Working for Process A?

```text
Process A (User Mode)
    |
    | syscall: read()
    v
Kernel (working for A)
    |
    | Timer Interrupt!
    v
kernelvec
    | Save interrupted registers on A's Kernel Stack
    v
kerneltrap()
    |
    | yield()
    v
Scheduler
    |
    | Choose Process B
    v
Process B runs
    |
    | Eventually A is scheduled again
    v
Resume A's unfinished Kernel work
    |
    | Return from kerneltrap()
    | Restore registers
    | sret
    v
Continue the interrupted Kernel instruction stream
```

This is one reason **each Process needs its own Kernel Stack**.

Process A's unfinished Kernel work remains on A's Kernel Stack while the CPU executes Process B.

A Timer Interrupt provides an opportunity for preemption, but it does not guarantee that a different Process will run. The Scheduler might choose A again.

Also, the Kernel does not continuously poll the clock. Hardware generates timer interrupts, and the Kernel handles them when interrupts are enabled.

---

## 5. Can the Scheduler Itself Be Interrupted?

Yes, a Timer Interrupt can occur while the Scheduler is executing, provided interrupts are enabled.

However, consider:

```c
if (which_dev == 2 && myproc() != 0)
    yield();
```

While the CPU is running `scheduler()` without a current Process:

```c
myproc() == 0
```

Therefore, the condition is false.

The Timer Interrupt Handler returns, and the CPU resumes executing the Scheduler.

**Why?**

Because `yield()` is designed to make the currently running Process give up the CPU.

If the CPU is already running the Scheduler, there is no current Process to yield.

---

## 6. Final Mental Model

Think of the CPU as a worker, the Process as a customer, and the Kernel as a set of privileged tools.

- **User Mode:** The worker performs the customer's work directly.
- **Kernel Mode:** The worker uses privileged tools to perform work for that customer.
- **Timer Interrupt:** A clock notifies the worker that it may be time to reconsider who gets the CPU.
- **`yield()`:** The current Process gives up the CPU.
- **Scheduler:** Decides which Process should run next.

Three key takeaways:

1. **Kernel Mode does not mean there is no current Process.** The Kernel often executes on behalf of a Process.
2. **Kernel Code can also be interrupted.** Timer Interrupts can interrupt both User Code and Kernel Code when permitted.
3. **Preemption must preserve unfinished work.** A Process's Kernel Stack allows the Kernel to resume its work after the Process is scheduled again.

> A Process is an execution context that can move between User Mode and Kernel Mode, be suspended, and later resume. Switching CPU privilege modes is not the same thing as switching Processes.